In [31]:
import os
# os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'

import torch
import numpy as np
import pandas as pd

import pyro

pyro.set_rng_seed(1)
assert pyro.__version__.startswith('1.9.1')

import sys

# Append a directory to the Python path
sys.path.append("./src")

pd.options.mode.chained_assignment = None

print(torch.backends.mps.is_available())
from src.fantasy_predictions import fantasy_predictions

from pybaseball import batting_stats

True


In [32]:
# load data
start_season =2020
end_season = 2025
qual = 100
bb_data = batting_stats(start_season=start_season, end_season=end_season, qual=qual)
print('data downloaded')

#%% define score map
score_map = {
    'R': 0.75,
    '1B': 1,
    '2B': 1.5,
    '3B': 2,
    'HR': 3,
    'RBI': 0.75,
    'BB':1, 
    'SO':-0.5,
    'HBP': 1,
    'SB': 1,
    'CS': -2,
    'GDP': -2,
}
bb_data['fpts'] = np.array([bb_data[k].to_numpy() * score_map[k] for k in score_map]).sum(axis = 0)
bb_data['core_fpts'] = np.array([bb_data[k].to_numpy() * score_map[k] for k in ['1B', '2B', '3B', 'HR', 'SO', 'BB', 'HBP']]).sum(axis = 0)


data downloaded


In [33]:
# preocess prediction data

test_data_t1 = bb_data[bb_data['Season'] == (end_season-1)]
test_data_t0 = bb_data[bb_data['Season'] == end_season]
test_data = test_data_t0[['Name', 'IDfg', 'Season', 'PA', 'Age']].set_index(['Name', 'IDfg']).join(
    test_data_t1[['Name', 'IDfg', 'Season', 'PA']].set_index(['Name', 'IDfg']),
    how = 'outer',
    lsuffix = f'{end_season}',
    rsuffix = f'{end_season-1}'
).fillna(300)

def average_pa(x0, x1):
    # use weighting and shrinkage by role
    mu = 0.7 * x0 + 0.3 * x1
    if mu > 520:
        out= 0.7 * mu + 0.3 * 650
    elif mu <= 520 and mu >350:
        out = 0.5 * mu + 0.5 * 485
    else:
        out = 0.4 * mu + 0.6 * 300
    return out

test_data['PA'] = test_data[['PA2025', 'PA2024']].apply(lambda row: average_pa(row['PA2025'], row['PA2024']), axis =1).round().astype(int)
test_data.reset_index(inplace = True)
test_data['Age'] = test_data['Age'] + 1

trn_data = bb_data[bb_data['IDfg'].isin(test_data['IDfg'].unique())]
test_data = test_data[test_data['IDfg'].isin(trn_data['IDfg'].unique())]

In [34]:
# generate predictions
import importlib

from src.fantasy_predictions import fantasy_predictions
fp = fantasy_predictions(trn_data = bb_data, test_data = test_data, device = 'mps')
fp.train(num_samples = 200, warmup_steps = 20)
fp.predict('mean')
forecast_results = fp.test_data.round()
forecast_results['fpts'] = np.array([forecast_results[k].to_numpy() * score_map[k] for k in ['1B', '2B', '3B', 'HR', 'SO', 'BB']]).sum(axis = 0)
#%% print results
res = forecast_results.groupby('Name').mean().join(
    forecast_results[['Name', 'fpts']].groupby('Name').std(),
    rsuffix = '_std'
).sort_values('fpts', ascending = False)
print(res.head(50))

train BB Model


Sample:  15%|█▍        | 32/220 [02:59, 10.39s/it, step size=4.14e-05, acc. prob=0.364]

KeyboardInterrupt: 

In [ ]:
# export results
res.to_csv('results/batting-predictions-sequential.csv', index=False)
print('Sequential predictions saved to results/batting-predictions-sequential.csv')